# Multiple environments in one MouseEnv

Pass a `list[EnvConfig]` to `make_env` to combine several different environment types in one `MouseEnv`. Each config expands its `num_envs` into independent single-env slots. All slots live in one flat list — `step()` takes and returns one entry per slot.

This notebook runs **CartPole-v1** (2 parallel slots) and **MountainCar-v0** (3 parallel slots) together, giving 5 slots total. It prints their specs and runs a short rollout loop.

## When to use multiple configs

The multi-env design is useful when your training loop needs to alternate between environments that have genuinely different observation and action spaces — for example, a curriculum that trains on easy envs and hard envs together, or a multi-task setup where each task is a different Gymnasium env.

Each config is fully independent: different `id`, `seed`, `num_envs`, reward shaping, and observation configuration. `MouseEnv` steps all slots sequentially and returns a single flat `list[dict]` of outputs — one entry per slot across all configs. Episode statistics accumulate in `env.tracker` and can be cleared with `env.tracker.clear()`.

## `env.names` and slot indexing

`env.names` is the full flat list of slot names across all configs:

```
("CartPole-v1_0", "CartPole-v1_1", "MountainCar-v0_0", "MountainCar-v0_1", "MountainCar-v0_2")
```

`env.num_envs` is the total slot count (5 here). `outputs[0]` and `outputs[1]` are CartPole slots; `outputs[2]`, `outputs[3]`, `outputs[4]` are MountainCar slots.

## Imports

In [1]:
from mouse_envs import EnvConfig, make_env

## Build a multi-environment

`make_env` accepts a plain list of configs. Each entry is independent — different ids, different `num_envs`, different action/observation spaces are all fine.

In [2]:
env = make_env([
    EnvConfig(id="CartPole-v1",    seed=0, num_envs=2, episodes_per_task=100),
    EnvConfig(id="MountainCar-v0", seed=1, num_envs=3, episodes_per_task=100),
])

## Inspect specs

`env.output_specs` and `env.input_specs` are flat lists — one entry per slot (5 total). `env.names` is the full flat list of slot names.

In [3]:
print(f"Total parallel slots : {env.num_envs}")
print(f"All names            : {env.names}")
print()
for name, ospec, ispec in zip(env.names, env.output_specs, env.input_specs):
    print(f"Slot {name}")
    print(f"  obs  dtype={ospec.observation.dtype}  shape={ospec.observation.shape}")
    print(f"  act  dtype={ispec.action.dtype}        shape={ispec.action.shape}")

Total parallel slots : 5
All names            : ('CartPole-v1#0', 'CartPole-v1#1', 'MountainCar-v0#0', 'MountainCar-v0#1', 'MountainCar-v0#2')

Slot CartPole-v1#0
  obs  dtype=torch.float32  shape=(4,)
  act  dtype=torch.int64        shape=()
Slot CartPole-v1#1
  obs  dtype=torch.float32  shape=(4,)
  act  dtype=torch.int64        shape=()
Slot MountainCar-v0#0
  obs  dtype=torch.float32  shape=(2,)
  act  dtype=torch.int64        shape=()
Slot MountainCar-v0#1
  obs  dtype=torch.float32  shape=(2,)
  act  dtype=torch.int64        shape=()
Slot MountainCar-v0#2
  obs  dtype=torch.float32  shape=(2,)
  act  dtype=torch.int64        shape=()


## Rollout loop

`env.sample_random_inputs()` returns `list[dict]` — one dict per slot (5 total).
`env.step(inputs)` returns a flat `list[dict]` of outputs — one entry per slot.

Episode statistics accumulate in `env.tracker`. Call `env.tracker.clear()` to reset
between evaluation runs. Slice by index range to separate CartPole from MountainCar results.

In [4]:
for step in range(300):
    env.step(env.sample_random_inputs())

# Episode stats accumulated in env.tracker automatically during the rollout
for slot_idx, rewards in enumerate(env.tracker.episode_cum_rewards):
    name = env.names[slot_idx]
    avg = sum(rewards) / len(rewards) if rewards else float("nan")
    print(f"{name:25s}  episodes={len(rewards):3d}  avg_return={avg:.2f}")

env.close()

CartPole-v1#0              episodes= 10  avg_return=27.80
CartPole-v1#1              episodes= 15  avg_return=18.93
MountainCar-v0#0           episodes=  1  avg_return=-200.00
MountainCar-v0#1           episodes=  1  avg_return=-200.00
MountainCar-v0#2           episodes=  1  avg_return=-200.00
